# ⚙️ Configuration & Export

How to configure parsing: mask proper nouns, remove broken words, and build multi-project datasets.

## 🎯 Goals of this notebook

1. **Understand RunConfig options** — learn every toggle the parser exposes
2. **Compare outputs** — see how dropping damaged signs or masking POS changes transliterations
3. **Build a multi-project dataset** — parse several corpora and combine into one DataFrame
4. **Export** — save combined data as JSONL and CSV for downstream use
5. **Quick analysis** — explore the combined dataset by project, provenance, and period

## Prerequisite: install the package

In [ ]:
# Install oracc-parser from PyPI
!pip install oracc-parser -q

## Prerequisite: set up the data directory

A DATA_DIR must be defined for the package to work correctly. This is where the data is stored after download from Zenodo or the live ORACC server.

**Option 1**: Colab

Set it up as a folder in your Google Drive (requires mounting Google Drive).

**Option 2**: Run Locally

Set it up anywhere you want on your computer.

Choose whichever option below you prefer (comment out the other one).

In [ ]:
# --- Option 1: persist downloaded data across Colab sessions using Google Drive ---
# Without this, data downloads to /content/oracc_data and is lost when the session ends.
# Uncomment the lines below to mount your Drive and store data there persistently.
#
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ["ORACC_DATA_DIR"] = "/content/drive/MyDrive/oracc_data"

# --- Option 2: run locally (not in Colab) ---
# If you are running this notebook locally, set the data directory here.
# Uncomment the lines below and set the path to where you want data stored.
#
# import oracc_parser.settings as settings
# from pathlib import Path
# settings.DATA_DIR = Path("/path/to/your/oracc_data")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 3.0 MB/s eta 0:00:00
Mounted at /content/drive


## 1. RunConfig options

These are RunConfig's parameters:

| Option | Default | What it does |
|---|---|---|
| `limit` | `None` | Only parse first N texts (good for testing) |
| `max_break_fraction` | `1.0` | **Word-level**: fraction of broken signs tolerated before a word is replaced with `X` in transliteration / normalization / lemmatization |
| `drop_missing` | `False` | **Sign-level**: replace signs marked `[x]` (completely broken) with `X` in **Unicode cuneiform output only** |
| `drop_damaged` | `False` | **Sign-level**: replace signs marked `⸢x⸣` (partially damaged) with `X` in **Unicode cuneiform output only** |
| `mask_pos` | `[]` | Replace words of certain POS with the tag name |

### ✅ Valid Config Values

When configuring a parser run with `RunConfig` (as seen in Notebook 01), you can filter by `languages` or `mask_pos`. The available values are exactly those listed in the reference tables below:

- **`mask_pos`**: Any POS tag from the `pos_tags` table below (e.g., `PN`, `DN`, `RN`, `N`, `V`).
- **`provenance` / `period`**: If filtering results programmatically by these fields in your analysis, use the `normalized_city` from the provenances table or the `period_name` from the periods table.

If you want to know what values are valid for `mask_pos`, `languages`, or `periods`, you can query the bundled reference data directly:

In [ ]:
from oracc_parser.pipeline import reference_data

# See valid POS tags for masking
pos_df = reference_data.get_pos_tags()
print('Valid POS tags (first 15):')
print(pos_df['POS-tag'].head(15).tolist())
print()

# See valid languages
lang_df = reference_data.get_languages()
print('Valid Languages:')
print(lang_df['lang'].tolist())


Valid POS tags (first 15):
[nan, 'N', 'u', 'V', 'PRP', 'n', 'PN', 'AJ', 'DET', 'DN', 'CNJ', 'MOD', 'X', 'REL', 'SN']

Valid Languages:
['akk', 'sux', 'akk-x-neoass', 'akk-x-neobab', 'akk-x-stdbab', 'akk-x-midass', 'akk-x-mbperi', 'akk-x-ltebab', 'qpc', 'akk-949', 'akk-x-oldbab', 'uga-040', 'peo', 'sux-x-emesal', 'arc', 'elx', 'akk-x-midbab', 'akk-x-oldass', 'uga', 'hit', 'qca', 'qcu-949', 'akk-x-oldakk', 'hit-946', 'qcu', 'qpe', 'xhu', 'xhu-040', 'akk-x-neoass-949', 'egy-020', 'akk-x-neobab-949', 'hit-040', 'xhu-946', 'arc-949', 'akk-x-stdbab-949']


## 2. Word-level vs sign-level break filtering

`RunConfig` exposes **two independent levels** of break filtering that operate on different granularities and affect different outputs.

### Word-level — `max_break_fraction` (0.0 – 1.0)

- Each word has a `break_perc`: the **fraction of its signs** that are broken or missing (averaged across all signs in the word).

$\text{break\_perc} = \frac{n_{\text{broken}} + 0.5 \cdot n_{\text{damaged}}}{n_{\text{total}}}$

- Words whose `break_perc` **exceeds** `max_break_fraction` are replaced wholesale with `X`.
- Affects → **transliteration**, **normalization**, **lemmatization**.
- `1.0` (default) → keep all words regardless of damage.
- `0.0` → replace any word that has even one broken sign.

### Sign-level — `drop_missing` / `drop_damaged`

- Operates **sign-by-sign**, not word-by-word.
- `drop_missing=True` replaces signs marked `[x]` (completely lost) with `X`.
- `drop_damaged=True` replaces signs marked `⸢x⸣` (partially legible) with `X`.
- Affects → **Unicode cuneiform output only**.

> ⚠️ **The two levels are independent and produce results that may not align.**
> A word kept intact in the transliteration (because its *average* damage is below `max_break_fraction`) may still have individual signs stripped from the Unicode cuneiform output by `drop_missing` / `drop_damaged`, and vice-versa. Do not expect the text outputs and the Unicode cuneiform to be aligned token-for-token when using these parameters.

In [ ]:
from oracc_parser import load_word_csvs_from_dir, parse_project_from_word_csvs, RunConfig, get_transliterations, get_unicode_texts
from oracc_parser.settings import WORD_CSV_DIR

PROJECT = "babcity" # change to another oracc project if desired

# Requires notebook 01 to have been run first — word CSVs must be downloaded locally
project_slug = PROJECT.replace("/", "-")
all_word_dfs = load_word_csvs_from_dir(WORD_CSV_DIR / project_slug, project=PROJECT)

# Change 2 to any number (or remove the slice) to use more or fewer texts
word_dfs = dict(list(all_word_dfs.items())[:2])
print(f"Using {len(word_dfs)} of {len(all_word_dfs)} texts from {PROJECT}")

# Strict word-level filtering: words with >30% broken signs → replaced with X
rec_strict = parse_project_from_word_csvs(PROJECT, word_dfs, config=RunConfig(max_break_fraction=0.3))

# Liberal (default): keep all words regardless of damage
rec_liberal = parse_project_from_word_csvs(PROJECT, word_dfs, config=RunConfig(max_break_fraction=1.0))

print("=== Transliteration with max_break_fraction=0.3 (strict) ===")
for _, r in get_transliterations(rec_strict).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Transliteration with max_break_fraction=1.0 (liberal, default) ===")
for _, r in get_transliterations(rec_liberal).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Unicode cuneiform is unaffected by max_break_fraction ===")
print("(drop_missing / drop_damaged control sign-level filtering here)")
for _, r in get_unicode_texts(rec_strict).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

11:52:04 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity


Using 2 of 224 texts from babcity


Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 30.97it/s]
11:52:04 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs
Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 30.65it/s]
11:52:04 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs


=== Transliteration with max_break_fraction=0.3 (strict) ===
  babcity_P261352:
    X X X X X X X X X
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ X
A ša₂ {m}{d}EN.LIL₂-TIN{iṭ} ša₂ ina IGI {m}ti-ri-ra-ka-am-ma
DUMU E₂ š
    tokens_without_broken: 63/88
  babcity_P285916:
    E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik
ša₂ DA E₂ {m}SILA-a-a
{m}ni-din-tu₄ DUMU {m}ši-rik A {m}e-gi₃-bi
a-na i-di E₂ a
    tokens_without_broken: 91/92

=== Transliteration with max_break_fraction=1.0 (liberal, default) ===
  babcity_P261352:
    1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ {[m}{d}EN-it-tan-nu]
A ša₂ {m}{d}EN.LIL₂-TIN{
    tokens_without_broken: 88/88
  babcity_P285916:
    E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik
ša₂ DA E₂ {m}SILA-a-a
{m}ni-din-tu₄ DUMU {m}ši-rik A {m}e-gi₃-bi
a-na i-di E₂ a
    tokens_without_broken: 92/92

=== Unicode cuneiform is unaffected by max_break_fraction ===
(drop_missing / drop_damaged control sign-level filtering here)
  babcity_P261352: 𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 

In [ ]:
# drop_missing / drop_damaged operate sign-by-sign on Unicode cuneiform only
# They have no effect on transliteration, normalization, or lemmatization
rec_drop_none    = parse_project_from_word_csvs(PROJECT, word_dfs, config=RunConfig(limit=2, fetch_translations=False, drop_missing=False, drop_damaged=False))
rec_drop_missing = parse_project_from_word_csvs(PROJECT, word_dfs, config=RunConfig(limit=2, fetch_translations=False, drop_missing=True,  drop_damaged=False))
rec_drop_both    = parse_project_from_word_csvs(PROJECT, word_dfs, config=RunConfig(limit=2, fetch_translations=False, drop_missing=True,  drop_damaged=True))

print("=== Unicode cuneiform — no filtering ===")
for _, r in get_unicode_texts(rec_drop_none).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

print()
print("=== Unicode cuneiform — drop_missing=True ===")
for _, r in get_unicode_texts(rec_drop_missing).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

print()
print("=== Unicode cuneiform — drop_missing=True, drop_damaged=True ===")
for _, r in get_unicode_texts(rec_drop_both).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 26.22it/s]
11:53:05 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs
Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 26.45it/s]
11:53:05 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs
Processing babcity: 100%|██████████| 2/2 [00:00<00:00, 28.69it/s]
11:53:06 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 2 tablets for babcity from word CSVs


=== Unicode cuneiform — no filtering ===
  babcity_P261352: 𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭
  babcity_P285916: 𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆
𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀
𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕𒁉
𒀀𒈾 𒄿𒁲 𒂍 𒀀𒈾 𒈬𒀭𒈾
𒌋𒐉 𒂆 𒆬𒌓 𒌓𒌑 𒀀𒈾 𒁹𒆠𒀭𒌓𒁷

=== Unicode cuneiform — drop_missing=True ===
  babcity_P261352: X X XX XX XX X X X XX
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 XXXXXX
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭
  babcity_P285916: 𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆
𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀
𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕𒁉
𒀀𒈾 𒄿𒁲 𒂍 𒀀𒈾 𒈬𒀭𒈾
𒌋𒐉 𒂆 𒆬𒌓 𒌓𒌑 𒀀𒈾 𒁹𒆠𒀭𒌓𒁷

=== Unicode cuneiform — drop_missing=True, drop_damaged=True ===
  babcity_P261352: X X XX XX XX X X X XX
𒁹𒁕𒊑𒅀X𒈲 𒈗 𒃻 XXXXXX
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭
  babcity_P285916: 𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆
𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀
𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕𒁉
𒀀𒈾 𒄿𒁲 𒂍 𒀀𒈾 𒈬𒀭𒈾
𒌋𒐉 𒂆 𒆬𒌓 𒌓𒌑 𒀀𒈾 𒁹𒆠𒀭𒌓𒁷


In [ ]:
from oracc_parser.pipeline import get_full_flat_table
from oracc_parser import load_word_csvs_from_dir, parse_project_from_word_csvs
from oracc_parser.settings import WORD_CSV_DIR
import pandas as pd

# Requires notebook 01 to have been run first — word CSVs must be downloaded locally
# Change these to any projects you want
PROJECTS = ["babcity", "borsippa"]
config = RunConfig(max_break_fraction=0.5, drop_missing=True)

all_dfs = []
for project in PROJECTS:
    print(f"Parsing {project}...")
    try:
        project_slug = project.replace("/", "-")
        all_word_dfs = load_word_csvs_from_dir(WORD_CSV_DIR / project_slug, project=project)
        word_dfs = dict(list(all_word_dfs.items())[:3])  # change 3 to use more or fewer texts
        records = parse_project_from_word_csvs(project, word_dfs, config=config)
        df = get_full_flat_table(records)
        all_dfs.append(df)
        print(f"  → {len(records)} tablets")
    except Exception as e:
        print(f"  → Error: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"\n✓ Combined: {len(combined)} tablets from {len(PROJECTS)} projects")
display(combined)


Parsing babcity...


12:04:54 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/babcity
Processing babcity: 100%|██████████| 3/3 [00:00<00:00, 35.02it/s]
12:04:54 | INFO    | oracc_parser | Processed 3 tablets for babcity from word CSVs
INFO:oracc_parser:Processed 3 tablets for babcity from word CSVs


  → 3 tablets
Parsing borsippa...


12:04:56 | INFO    | oracc_parser | Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
INFO:oracc_parser:Loaded 224 word CSV(s) from /content/drive/MyDrive/oracc_data/oracc_csvs/borsippa
Processing borsippa: 100%|██████████| 3/3 [00:00<00:00, 32.93it/s]
12:04:56 | INFO    | oracc_parser | Processed 3 tablets for borsippa from word CSVs
INFO:oracc_parser:Processed 3 tablets for borsippa from word CSVs


  → 3 tablets

✓ Combined: 6 tablets from 2 projects


,id,project,text_id,genre,archive,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,babcity_P261352,babcity,P261352,Legal,Murašû Archive,Nippur,achaemenid,-547,-331,X X X X X X X X X\n{m}da-ri-ia-⸢a⸣-muš LUGAL š...,X X X X X X X X X\nDarius šarri ša X\nmāru ša ...,X X X X X X X X X\nDarius šarru ša X\nmāru ša ...,X X XX XX XX X X X XX\n𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 XXXXXX\n𒀀 𒃻 ...,,88,64
1,babcity_P285916,babcity,P285916,Legal,Egibi Archive,Babylon,achaemenid,-547,-331,E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik\nša₂ ...,bītu ša Nidinti-Bel māri Širku\nša ṭāh bīt Suq...,bītu ša Nidinti-Bel māru Širku\nša ṭāh bītu Su...,𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆\n𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀\n𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕...,,92,92
2,babcity_P295135,babcity,P295135,Legal,Ea-ilutu-bani Archive,Borsippa,achaemenid,-547,-331,E₂-⸢su⸣ ša₂ {e₂}a-su-up ša₂ {m}⸢x⸣-[...]\nA-šu...,bīssu ša asup ša X\nmārišu ša Luṣi-ana-nur-Mar...,bītu ša asuppu ša X\nmāru ša Luṣi-ana-nur-Mard...,𒂍𒋢 𒃻 𒂍𒀀𒋢𒌒 𒃻 𒁹XX\n𒀀𒋙 𒃻 𒁹𒇻𒍮𒀀𒈾𒂟𒀭𒀫𒌓 X\n𒀀𒈾 𒈬𒀭𒈾 𒈫 𒂆 ...,,84,70
3,borsippa_P202111,borsippa,P202111,Legal,Ilia Archive,Borsippa,achaemenid,-547,-331,2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu...,UNK UNK ūmu maštīti ša Ayyaru UNK UNK ūmu\nmaš...,UNK UNK ūmu maštītu ša Ayyaru UNK UNK ūmu\nmaš...,𒈫 𒈦 𒌓𒈬 𒈦𒋾𒋾 𒃻 𒌗𒄞 𒈫 𒈦 𒌓𒈬\nX𒋾𒋾 𒌗𒆥 𒈫 𒈦 𒌓𒈬 𒆏𒊑 𒃻 𒌗𒃶\...,,102,91
4,borsippa_P202351,borsippa,P202351,Legal,Ilia Archive,Borsippa,Neo-Babylonian,-626,-539,X X X {m}ṣil-la-a A {m}DINGIR-⸢ia⸣\nX X lib₃]-...,X X X Ṣilla mār Ilia\nX X libbišu Kalba qallaš...,X X X Ṣilla māru Ilia\nX X libbu Kalba qallu\n...,XXXX XX X 𒁹𒉣𒆷𒀀 𒀀 𒁹𒀭𒅀\nX XX X𒁉𒋙 𒁹𒆗𒁀𒀀 𒇽𒃲𒆷X\nXX 𒇽...,,157,96
5,borsippa_P202504,borsippa,P202504,Legal,Ahiya’ūtu Archive,Borsippa,achaemenid,-547,-331,X u₄]-mu tar-din-nu {iti}ŠE TA UD [x]-KAM\nX ⸢...,X ūmu tardinnu Addaru ultu ūm UNK\nX ūm UNK UN...,X ūmu tardennu Addaru ištu ūmu UNK\nX ūmu UNK ...,X X𒈬 𒋻𒁷𒉡 𒌗𒊺 𒋫 𒌓 X𒄰\nXX 𒌓 𒌋𒐋𒄰 𒐈 𒌓𒈬 𒋾𒅆𒆕𒈨𒌍\nX𒁈 𒌓 ...,,247,181


In [ ]:
from oracc_parser.settings import OUTPUT_DIR

# Export the combined dataset
out = OUTPUT_DIR
out.mkdir(parents=True, exist_ok=True)

# change the file names here, if you want
path_jsonl = out / "combined_dataset.jsonl"
path_csv = out / "combined_dataset.csv"

combined.to_json(path_jsonl, orient="records", lines=True, force_ascii=False)
combined.to_csv(path_csv, index=False)

print(f"Saved to:")
print(f"  {path_jsonl}  ({path_jsonl.stat().st_size/1024:.1f} KB)")
print(f"  {path_csv}  ({path_csv.stat().st_size/1024:.1f} KB)")
print(f"\n📁 All output files in {out}:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved to:
  /content/drive/MyDrive/oracc_data/output/combined_dataset.jsonl  (24.5 KB)
  /content/drive/MyDrive/oracc_data/output/combined_dataset.csv  (22.8 KB)

📁 All output files in /content/drive/MyDrive/oracc_data/output:
  babcity.csv                               16.1 KB
  babcity.jsonl                             17.4 KB
  combined_dataset.csv                      22.8 KB
  combined_dataset.jsonl                    24.5 KB


## 5. Quick analysis of the data

In [ ]:
print("Texts by project:")
print(combined["project"].value_counts().to_string())

print("\nTexts by provenance:")
print(combined["provenance"].value_counts().to_string())

print("\nTexts by period:")
print(combined["period"].value_counts().to_string())

print(f"\nAvg tokens per text: {combined['total_tokens'].mean():.0f}")
print(f"Total tokens: {combined['total_tokens'].sum()}")

Texts by project:
project
babcity     3
borsippa    3

Texts by provenance:
provenance
Borsippa    4
Nippur      1
Babylon     1

Texts by period:
period
achaemenid        5
Neo-Babylonian    1

Avg tokens per text: 128
Total tokens: 770
